# DATA 298B Agentic AML Evaluation

Runs the slower tool-using benchmark for the provisional top two models: Gemma and Qwen. It does not train models or rerun the fast comparison.

In [ ]:
# ---- Edit only this cell ----
REPO_URL = 'https://github.com/aryaMehta26/Graph-Based-Fraud-Intelligence-Platform.git'
REPO_BRANCH = 'feature/298b-backend'
HELDOUT_DATASET = '/content/drive/MyDrive/DATA298B/heldout/aml_heldout.jsonl'
ADAPTER_ROOT = '/content/drive/MyDrive/DATA298B/adapters'
OUTPUT_ROOT = '/content/drive/MyDrive/DATA298B/final_benchmark'
MODELS = ['gemma', 'qwen']
N_LEGITIMATE = 10
N_SUSPICIOUS = 10
MAX_TOOL_CALLS = 3
MAX_TOKENS = 700
SEED = 42

In [ ]:
# Mount Drive and pull the latest branch.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess
REPO_DIR = Path('/content/Graph-Based-Fraud-Intelligence-Platform')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('repository ready:', REPO_DIR)

In [ ]:
# Install dependencies and verify the GPU.
%pip install -q -r requirements.txt
%pip install -q accelerate bitsandbytes sentencepiece huggingface_hub
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Colab GPU runtime before starting agentic evaluation.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

In [ ]:
# Secure Hugging Face login.
from huggingface_hub import login
login(add_to_git_credential=False)

In [ ]:
# Load the same held-out cases used in the fast comparison.
import json
heldout_path = Path(HELDOUT_DATASET)
records = [json.loads(line) for line in heldout_path.read_text().splitlines() if line.strip()]
legitimate = [r for r in records if r['ground_truth']['decision'] == 'LEGITIMATE'][:N_LEGITIMATE]
suspicious = [r for r in records if r['ground_truth']['decision'] == 'SUSPICIOUS'][:N_SUSPICIOUS]
selected_records = legitimate + suspicious
if len(selected_records) != N_LEGITIMATE + N_SUSPICIOUS:
    raise ValueError('The held-out dataset does not contain enough legitimate and suspicious cases.')
print('agentic cases:', len(selected_records))

In [ ]:
# Run Gemma and Qwen sequentially. This is the long-running cell.
import gc
import yaml
from src.aml.backend import TransformersBackend
from src.aml.agent import AMLAgent
from src.aml.schemas import InvestigationCase
from src.evaluation.benchmark_runner import run as evaluate_traces
registry = yaml.safe_load((REPO_DIR / 'configs/models/registry.yaml').read_text())
root = Path('/content/drive/MyDrive/DATA298B')
for MODEL in MODELS:
    print(f'\n===== {MODEL.upper()} =====', flush=True)
    adapter_dir = root / 'adapters' / f'{MODEL}_full_grounded'
    model_output = root / 'final_benchmark' / f'agentic_{MODEL}'
    trace_dir = model_output / 'traces'
    trace_jsonl = model_output / 'agentic_traces.jsonl'
    model_output.mkdir(parents=True, exist_ok=True)
    trace_dir.mkdir(parents=True, exist_ok=True)
    spec = registry[MODEL]
    backend = TransformersBackend(spec['model_id'], revision=spec['revision'], adapter_path=str(adapter_dir), model_kwargs={'load_in_4bit': True, 'device_map': 'auto'})
    completed = {}
    if trace_jsonl.exists():
        completed = {json.loads(line)['case_id']: True for line in trace_jsonl.read_text().splitlines() if line.strip()}
    mode = 'a' if trace_jsonl.exists() else 'w'
    with trace_jsonl.open(mode) as output:
        for index, record in enumerate(selected_records, 1):
            if record['case_id'] in completed:
                print(f'{MODEL}: {index}/{len(selected_records)} {record["case_id"]}: already complete', flush=True)
                continue
            case = InvestigationCase(case_id=record['case_id'], context=record['context'], ground_truth=record['ground_truth'], split=record.get('split', 'test'))
            print(f'{MODEL}: {index}/{len(selected_records)} {case.case_id}', flush=True)
            trace = AMLAgent(backend, max_tool_calls=MAX_TOOL_CALLS, artifact_dir=str(trace_dir)).investigate(case, model_family=MODEL, variant='finetuned_agentic', seed=SEED, inference_config={'max_tokens': MAX_TOKENS, 'temperature': 0.0})
            output.write(json.dumps(trace, default=str) + '\n')
            output.flush()
    metrics = evaluate_traces(trace_jsonl, model_output / 'metrics')
    print(json.dumps(metrics, indent=2, default=str))
    del backend
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# Review the two agentic metric files.
for MODEL in MODELS:
    path = Path(OUTPUT_ROOT) / f'agentic_{MODEL}' / 'metrics' / 'benchmark_metrics.json'
    print('\n', MODEL.upper())
    print(path)
    print(json.dumps(json.loads(path.read_text()), indent=2))

In [ ]:
# Run only after both models and metrics are saved.
from google.colab import runtime
runtime.unassign()